# Silence detection QA

Listen to detected silent / non-silent sections after optional **denoise** and **karaoke vocal separation**.

Pipeline: raw → denoise → (optional) karaoke separation → silence detection

```bash
python eval_silence_detection.py --audio-variant denoised
python eval_silence_detection.py --audio-variant all
```

Then inspect waveform shading and play back detected sections below.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display
from idtap import Piece, SwaraClient

from silence.silence_audio_prep import load_variant_audio
from silence.silence_detection import SilentSegment, segment_audio

OUTPUT_DIR = PROJECT_ROOT / "output"
DENOISE_ROOT = OUTPUT_DIR / "denoised"
METADATA_PATH = OUTPUT_DIR / "cnn_dataset" / "all" / "metadata.csv"
SUMMARY_PATH = OUTPUT_DIR / "silence_detection" / "raw_baseline" / "summary.json"
SR = 22050
PIECE_ID = "6824de49abc4705438ce918b"

# Audio prep + detection controls
AUDIO_VARIANT = "denoised"  # "raw", "denoised", or "vocals"
METHOD = "rms"  # "rms" or "librosa_split"
SKIP_DENOISE = False  # True to use cached stems only
FORCE_VOCAL_SEPARATION = False  # True to run karaoke even on instrumental pieces
MAX_SECTIONS = 12
MIN_SECTION_DURATION = 0.25


In [ ]:
metadata = pd.read_csv(METADATA_PATH)
metadata["piece_id"] = metadata["piece_id"].astype(str)
piece_rows = metadata[metadata["piece_id"] == PIECE_ID]
piece_title = str(piece_rows.iloc[0]["piece_title"])

client = SwaraClient()
piece_obj = Piece.from_json(client.get_piece(PIECE_ID))

y, stem_path, prep_meta = load_variant_audio(
    piece_id=PIECE_ID,
    piece_title=piece_title,
    audio_dir=OUTPUT_DIR,
    denoise_root=DENOISE_ROOT,
    variant=AUDIO_VARIANT,
    sr=SR,
    piece_obj=piece_obj,
    skip_denoise=SKIP_DENOISE,
    force_vocal_separation=FORCE_VOCAL_SEPARATION,
)
sr = SR
duration = len(y) / sr

summary_path = OUTPUT_DIR / "silence_detection" / f"{AUDIO_VARIANT}_baseline" / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    best_rms = summary["best_rms_params"]
    best_librosa = summary["best_librosa_params"]
else:
    summary_path = SUMMARY_PATH
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    best_rms = summary.get("best_rms_params", {"threshold_db": -40.0, "hop_length": 512, "min_duration": 0.1})
    best_librosa = summary.get("best_librosa_params", {"top_db": 40.0, "hop_length": 512, "min_duration": 0.1})

segments_rms = segment_audio(y, sr, "rms", **best_rms)
segments_librosa = segment_audio(y, sr, "librosa_split", **best_librosa)
segments_by_method = {"rms": segments_rms, "librosa_split": segments_librosa}

print(f"Piece: {piece_title}")
print(f"Audio variant: {AUDIO_VARIANT}")
print(f"Stem: {stem_path}")
print(f"Prep: {prep_meta}")
print(f"Duration: {duration:.1f}s")
print(f"Summary: {summary_path if summary_path.exists() else 'defaults'}")
for method_name, segments in segments_by_method.items():
    n_silent = sum(1 for s in segments if s.is_silent)
    print(f"{method_name}: {len(segments)} sections ({n_silent} silent, {len(segments) - n_silent} non-silent)")

In [ ]:
def plot_segmentation(
    y: np.ndarray,
    sr: int,
    segments: list[SilentSegment],
    *,
    title: str,
    max_seconds: float | None = 120.0,
) -> None:
    end_time = min(len(y) / sr, max_seconds) if max_seconds is not None else len(y) / sr
    end_sample = int(end_time * sr)
    y_plot = y[:end_sample]
    times = np.linspace(0, end_time, num=len(y_plot))

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(times, y_plot, color="black", linewidth=0.5, alpha=0.8)

    for segment in segments:
        if segment.end <= 0 or segment.start >= end_time:
            continue
        color = "#cccccc" if segment.is_silent else "#ffd966"
        ax.axvspan(
            max(0.0, segment.start),
            min(end_time, segment.end),
            color=color,
            alpha=0.35,
            linewidth=0,
        )

    ax.set_xlim(0, end_time)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


variant_label = AUDIO_VARIANT
plot_segmentation(
    y,
    sr,
    segments_rms,
    title=f"[{variant_label}] RMS segmentation (gray=silent, yellow=non-silent)",
)
plot_segmentation(
    y,
    sr,
    segments_librosa,
    title=f"[{variant_label}] librosa.effects.split segmentation",
)

In [ ]:
def play_detected_sections(
    y: np.ndarray,
    sr: int,
    segments: list[SilentSegment],
    *,
    method_name: str,
    max_sections: int = MAX_SECTIONS,
    min_duration: float = MIN_SECTION_DURATION,
) -> None:
    eligible = [segment for segment in segments if segment.duration >= min_duration]
    print(
        f"[{AUDIO_VARIANT}] {method_name}: playing up to {max_sections} of "
        f"{len(eligible)} sections (min duration {min_duration:.2f}s)"
    )

    for idx, segment in enumerate(eligible[:max_sections], start=1):
        start_sample = int(segment.start * sr)
        end_sample = int(segment.end * sr)
        clip = y[start_sample:end_sample]
        label = "silent" if segment.is_silent else "non-silent"
        print(
            f"\n[{idx}] {label} | {segment.start:.2f}s -> {segment.end:.2f}s "
            f"({segment.duration:.2f}s)"
        )
        display(Audio(clip, rate=sr))


if METHOD not in segments_by_method:
    raise ValueError(f"Unknown METHOD={METHOD!r}; choose from {list(segments_by_method)}")

play_detected_sections(
    y,
    sr,
    segments_by_method[METHOD],
    method_name=METHOD,
    max_sections=MAX_SECTIONS,
    min_duration=MIN_SECTION_DURATION,
)